In [10]:
import requests
from xml.etree import ElementTree
import json
import pprint
import time
import aiohttp
import asyncio

In [ ]:
# installed PyTorch
# ! pip install torch torchvision

   ---------------------------------------- 0.0/109.3 MB ? eta -:--:--
   - -------------------------------------- 3.4/109.3 MB 17.0 MB/s eta 0:00:07
   --- ------------------------------------ 10.5/109.3 MB 26.0 MB/s eta 0:00:04
   ------- -------------------------------- 19.7/109.3 MB 31.8 MB/s eta 0:00:03
   ---------- ----------------------------- 28.6/109.3 MB 34.7 MB/s eta 0:00:03
   -------------- ------------------------- 39.8/109.3 MB 38.4 MB/s eta 0:00:02
   ------------------ --------------------- 51.4/109.3 MB 41.0 MB/s eta 0:00:02
   ----------------------- ---------------- 63.4/109.3 MB 43.0 MB/s eta 0:00:02
   --------------------------- ------------ 73.9/109.3 MB 43.8 MB/s eta 0:00:01
   ------------------------------ --------- 84.4/109.3 MB 44.3 MB/s eta 0:00:01
   ---------------------------------- ----- 93.1/109.3 MB 43.9 MB/s eta 0:00:01
   ------------------------------------ -- 102.8/109.3 MB 44.0 MB/s eta 0:00:01
   --------------------------------------  109.1/1

In [ ]:
# insured torch installed correctly
# asked ChatGPT how to guarentee it worked
import torch
print(torch.__version__)
print("CUDA available:", torch.cuda.is_available())


2.9.0+cpu
CUDA available: False


In [ ]:
# installed huggingfacefrom transformers import AutoTokenizer, AutoModel

# ! pip install transformers

   ---------------------------------------- 0.0/12.0 MB ? eta -:--:--
   -------------- ------------------------- 4.5/12.0 MB 23.0 MB/s eta 0:00:01
   -------------------------------------- - 11.5/12.0 MB 28.8 MB/s eta 0:00:01
   ---------------------------------------- 12.0/12.0 MB 23.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/566.1 kB ? eta -:--:--
   --------------------------------------- 566.1/566.1 kB 17.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 32.2 MB/s eta 0:00:00

   ---------- ----------------------------- 1/4 [huggingface-hub]
   ---------- ----------------------------- 1/4 [huggingface-hub]
   ---------- ----------------------------- 1/4 [huggingface-hub]
   ---------- ----------------------------- 1/4 [huggingface-hub]
   ---------- ----------------------------- 1/4 [huggingface-hub]
   ------------------------------ --------- 3/4 [transformers]
   

In [4]:
from transformers import AutoTokenizer, AutoModel

# load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained('allenai/specter')
model = AutoModel.from_pretrained('allenai/specter')

tokenizer_config.json:   0%|          | 0.00/321 [00:00<?, ?B/s]

c:\Users\carol\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\carol\.cache\huggingface\hub\models--allenai--specter. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [11]:
# pull in 1000 alz articles
base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
params = {
    "db": "pubmed",
    "term": "Alzheimers AND 2024[pdat]",
    "retmax": "1000",
    "retmode": "xml"
}

# Send request
response = requests.get(base_url, params=params)
root = ElementTree.fromstring(response.text)

# Extract article IDs
alz_ids = [id_elem.text for id_elem in root.findall(".//Id")]
print(f"Found {len(alz_ids)} Alzheimers articles.")

Found 1000 Alzheimers articles.


In [12]:
# pull in 1000 cancer paper

base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
params = {
    "db": "pubmed", # from PubMed
    "term": "cancer AND 2024[pdat]", #cancer
    "retmax": "1000", # only send 1000 articles
    "retmode": "xml" # sends in xml format
}

response = requests.get(base_url, params=params)
root = ElementTree.fromstring(response.text)

cancer_ids = [id_elem.text for id_elem in root.findall(".//Id")]
print(f"Found {len(cancer_ids)} cancer articles.")

Found 1000 cancer articles.


In [13]:
all_pmids = cancer_ids + alz_ids

# Optionally remove duplicates while preserving order
all_pmids_unique = list(dict.fromkeys(all_pmids))

print(f"Total combined PMIDs: {len(all_pmids)}")
print(f"Unique PMIDs: {len(all_pmids_unique)}")

Total combined PMIDs: 2000
Unique PMIDs: 1996


In [ ]:


import tqdm

# we can use a persistent dictionary (via shelve) so we can stop and restart if needed
# alternatively, do the same but with embeddings starting as an empty dictionary

embeddings = {}
for pmid, paper in tqdm.tqdm(papers.items()):
    data = [paper["ArticleTitle"] + tokenizer.sep_token + get_abstract(paper)]
    inputs = tokenizer(
        data, padding=True, truncation=True, return_tensors="pt", max_length=512
    )
    result = model(**inputs)
    # take the first token in the batch as the embedding
    embeddings[pmid] = result.last_hidden_state[:, 0, :].detach().numpy()[0]

# turn our dictionary into a list
embeddings = [embeddings[pmid] for pmid in papers.keys()]

NameError: name 'papers' is not defined